<a href="https://colab.research.google.com/github/Aivon99/BigDataAndTextMiningProject/blob/Ivo/src/eval/Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Downloading Libraries and Imports

In [ ]:
!rm -rf /usr/local/lib/python3.13/dist-packages/~orch*
!pip install -q --upgrade "pillow<11.0.0" torch torchvision transformers>=4.45.0 accelerate torchao scikit-learn tqdm chess cairosvg python-Levenshtein datasets peft huggingface_hub python-dotenv
# Not upgrading torchaudio isn't enough -- Colab preinstalls it built against a
# different CUDA version than the freshly-upgraded torch above, and merely
# leaving it alone still makes transformers crash when it tries to import it
# (this project has no audio functionality, so it's safe to remove outright).
!pip uninstall -y torchaudio -q

# Standard library imports
import json
import os
import subprocess
import sys
from pathlib import Path

# Third-party library imports
import numpy as np
from functools import partial
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from huggingface_hub import login
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from PIL import Image
from tqdm import tqdm
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)


Setting up environment

In [ ]:
# Project configuration dictionary
CONFIG = {
    "colab": True,
    "branch": "Ivo",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

# 1. Setup repository path and handle cloning safely
repo_root = Path(CONFIG["repo_dir"])

if CONFIG["colab"]:
    # If the repository folder already exists, reference it safely
    if repo_root.exists():
        print(f"Repository directory already exists at: {repo_root}")
    else:
        auth_url = "https://"
        repo_url = f"{auth_url}github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

        print(f"Cloning repository from {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
else:
    repo_root = Path(".").resolve().parent.parent


print(f"Setup Complete. REPO_ROOT: {repo_root}")

# Check GPU and device availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Configure system paths for absolute module imports
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "src" / "data"))
sys.path.insert(0, str(repo_root / "src" / "eval"))

# 4. Import custom project modules cleanly
from data.generation import (
    build_sample,
    generate_dataset,
)

from data.utilities import (
    load_lichess_csv,
    upload_dataset_to_hub,
    authenticate_hf,
)

from eval.utilities import (
   calculate_fen_exact_match,
   calculate_levenshtein_metrics,
   calculate_square_by_square_accuracy,
   evaluate_chessboard_model_task_1,
   preprocess_function,
   get_patch_reordering_indices,
   reorder_chessboard_image,
   apply_patch_permutation,
   finetune_and_push_chessboard_model,
   find_resumable_checkpoint,
   Qwen35VisionDataCollator,
)

from training.learned_reordering import train_with_learned_reordering

print("All custom modules and eval utilities imported successfully!")

Dowloading dataset for task 1 from HuggingFace repo

In [ ]:
# Authenticate with Hugging Face:
# - Colab: reads a token from a Colab secret named HF_TOKEN (key icon, left sidebar)
# - Local: reads HF_TOKEN from a repo-root .env file, or the environment
# - Falls back to an interactive login prompt if neither is found
hf_token = None
if CONFIG["colab"]:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
else:
    from dotenv import load_dotenv
    load_dotenv(repo_root / ".env")
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub using HF_TOKEN.")
else:
    print("No HF_TOKEN found in secrets/.env/environment — falling back to interactive login.")
    login()

dataset_name = "bdatm-project/dataset_task1"
print(f"Downloading dataset '{dataset_name}'...")

dataset_task1 = load_dataset(dataset_name)

print("\nDataset loaded successfully!")
print(dataset_task1)
print("\nStructure sample of train split:")
print(dataset_task1["train"][0])

## Vanilla Model

Loading model

In [ ]:
model_id = "Qwen/Qwen3.5-0.8B"
print(f"Loading model {model_id}...")

model_vanilla = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)
print("model and Processor loaded correctly!")

Test with baseline on a single sample

In [ ]:
# Grab the first test sample
test_sample = dataset_task1["test"][0]

fen = test_sample["fen"]
task_prompt = test_sample["prompt"]
ground_truth_fen = test_sample["target"]
sample_id = test_sample["sample_id"]
board_image = test_sample["image"]

print(f"Sample ID: {sample_id}")
print(f"FEN: {fen}")
print(f"Prompt provided to the model:\n{task_prompt}\n")
print(f"Real FEN (Ground Truth): {ground_truth_fen}\n")

# Prepare the multimodal input format for the model
chat_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": board_image},
            {"type": "text", "text": task_prompt},
        ]
    }
]

# Apply the processor's chat template
formatted_text = processor.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)

# Tokenize inputs and move them to the GPU device
model_inputs = processor(
    text=[formatted_text],
    images=board_image,
    padding=True,
    return_tensors="pt"
).to(model_vanilla.device)

# Generate the zero-shot prediction
print("Generating zero-shot prediction...")
with torch.no_grad():
    output_token_ids = model_vanilla.generate(**model_inputs, max_new_tokens=128)

# Trim prompt tokens from the generated output
trimmed_output_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output_token_ids)
]
predicted_fen_string = processor.batch_decode(
    trimmed_output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(f"Predicted FEN (Zero-Shot): {predicted_fen_string.strip()}")

Testing vanilla model on the whole dataset using the function *evaluate_chessboard_model_task_1*

In [ ]:
# Call the evaluation function for the vanilla model
vanilla_results_df, vanilla_summary_df = evaluate_chessboard_model_task_1(
    model=model_vanilla,
    processor=processor,
    dataset_split=dataset_task1["test"],
    model_name="Vanilla Qwen2.5-VL (Zero-Shot)"
)

# Initialize the global comparison DataFrame with the vanilla results
all_models_results = vanilla_summary_df

print("\nExample of results obtained form the evaluation:")
display(vanilla_results_df.head())

print("\nComparative Summary DataFrame (all_models_results):")
display(all_models_results)

## Vanilla + LoRA

Preprocessing dataset and finetuning

In [ ]:
# 1. Preprocessing dataset
print("Applying preprocessing to datasets...")
tokenized_train = dataset_task1["train"].map(
    partial(preprocess_function, processor=processor),
    remove_columns=dataset_task1["train"].column_names,
)
tokenized_val = dataset_task1["validation"].map(
    partial(preprocess_function, processor=processor),
    remove_columns=dataset_task1["validation"].column_names,
)

# 2. Configure PEFT and LoRA parameters
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
)

# Apply LoRA to model vanilla and saving the new one into lora_model
lora_model = get_peft_model(model_vanilla, peft_config)
lora_model.print_trainable_parameters()

# 3. Resolve the target Hub repo now (needed before training starts) and
# check whether an earlier, interrupted run already left a resumable
# checkpoint there (e.g. after a Colab disconnect wiped the local runtime).
hf_org_prefix = "bdatm-project"
repo_id_standard = f"{hf_org_prefix}/qwen-task1-standard-lora"
resume_checkpoint = find_resumable_checkpoint(repo_id_standard)

# 4. Define Training Arguments. Trains for up to 10 epochs, but relies on
# the validation set (via EarlyStoppingCallback below) to stop once
# eval_loss stops improving, and keeps the best-performing checkpoint
# (by validation loss) rather than just whichever epoch finishes last.
# push_to_hub + hub_strategy="checkpoint" uploads a fully resumable
# checkpoint (optimizer/scheduler/RNG state included) after every epoch,
# so a Colab disconnect at epoch i loses at most that epoch's progress.
training_args = TrainingArguments(
    output_dir="./qwen_task1_lora_output",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    push_to_hub=True,
    hub_model_id=repo_id_standard,
    hub_strategy="checkpoint",
    fp16=True,
    remove_unused_columns=False,
    report_to="none",
)

# 5. Initialize the Trainer using 'lora_model', with early stopping driven
# by the validation set (stops if eval_loss doesn't improve for 2 epochs).
# data_collator batches Qwen's per-sample image_grid_thw/pixel_values
# tensors correctly (torch.cat along dim 0) -- the default collator can't,
# and silently produces malformed batches that only surface later inside
# the vision tower as opaque IndexErrors.
data_collator = Qwen35VisionDataCollator(processor=processor)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# 6. Start (or resume) Fine-Tuning
if resume_checkpoint:
    print(f"Resuming LoRA Supervised Fine-Tuning from {resume_checkpoint}...")
else:
    print("Starting LoRA Supervised Fine-Tuning...")
trainer.train(resume_from_checkpoint=resume_checkpoint)

Saving model on hugging face

In [ ]:
# 6. Save weights to Hugging Face folder
hf_org_prefix = "bdatm-project"
repo_id_standard = f"{hf_org_prefix}/qwen-task1-standard-lora"

print(f"Pushing standard LoRA model and processor to Hugging Face Hub: {repo_id_standard}...")

trainer.model.push_to_hub(
    repo_id_standard,
    commit_message="Training complete for standard LoRA baseline (raster-scan)"
)
processor.push_to_hub(
    repo_id_standard
)

print("Fine-tuning completed and weights successfully uploaded to Hugging Face Hub!")

Loading model and evaluation

In [ ]:
# 7. Loading Model from hugging face and evaluate model using predefined functions
print("\nLoading standard LoRA model from Hugging Face for evaluation...")

# Load fresh base model instance
base_eval_model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3.5-0.8B",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Load the LoRA weights directly from the Hub repository
standard_lora_eval_model = PeftModel.from_pretrained(base_eval_model, repo_id_standard)

print("Evaluating the Hugging Face LoRA model on the test set...")
lora_results_df, lora_summary_df = evaluate_chessboard_model_task_1(
    model=standard_lora_eval_model,
    processor=processor,
    dataset_split=dataset_task1["test"],
    model_name="Qwen + LoRA Fine-Tuning (from HF)"
)

# Append the new metrics to the global comparison DataFrame
all_models_results = pd.concat([all_models_results, lora_summary_df], ignore_index=True)

print("\nUpdated Comparative Summary Table (all_models_results):")
display(all_models_results)

## Reordering patches - Advanced models

Checking function *get_patch_reordering_indices()*

In [ ]:
for strat in ["raster", "zigzag", "spiral", "file_wise"]:
    order_map = get_patch_reordering_indices(strategy=strat)
    print(f"Strategy '{strat}' first 10 patch indices: {order_map[:10]}")

Let's evalaute the model we created before using three different patch ordering: zigzag, spiral and file-wise. We'll use the function ***reorder_chessboard_image*** defined in *src/eval/utilities.py*

In [ ]:
# ==========================================
# Training-Free Benchmark Loop for REOrder
# ==========================================

# Define the strategies you want to benchmark (as outlined in the project specs)
strategies_to_test = ["zigzag", "spiral", "file_wise"]

# Choose the model to test (we use the fine-tuned LoRA model)
model_to_evaluate = lora_model  # You can switch to 'model' if you want to test the vanilla version

for strat in strategies_to_test:
    print(f"\nEvaluating strategy (Training-Free): {strat.upper()}...")

    # 1. Apply the reordering function to the images in the test set
    reordered_test_split = dataset_task1["test"].map(
        lambda sample: {
            "image": reorder_chessboard_image(sample["image"], strategy=strat, grid_size=8)
        }
    )

    # 2. Run the evaluation function on the reordered test split
    strat_results_df, strat_summary_df = evaluate_chessboard_model_task_1(
        model=model_to_evaluate,
        processor=processor,
        dataset_split=reordered_test_split,
        model_name=f"Qwen + LoRA ({strat.capitalize()} - TF)"
    )

    # 3. Append the results to your global comparison table
    all_models_results = pd.concat([all_models_results, strat_summary_df], ignore_index=True)

print("\n--- Final Comparative Summary Table (Including Training-Free Strategies) ---")
display(all_models_results)

As highlighted in the summary table, applying unconventional patch reordering strategies (such as Zigzag, Spiral, or File-wise) in a "Training-Free" (TF) manner leads to a performance drop, resulting in an increased Character Error Rate (CER) and Levenshtein distance compared to the standard raster-scan baseline.

To truly reap the benefits of the REOrder methodology, we must proceed with Supervised Fine-Tuning (SFT) directly on the pre-reordered dataset. This will allow the model to adapt its weights and attention layers to the new spatial serialization strategy.

Finetuning the new models on the dataset using function **finetune_and_push_chessboard_model()** defined in *src/eval/utilities.py*. This function directly upload the models on hugging face

In [ ]:
strategies_to_train = ["zigzag", "spiral", "file_wise"]

# Dictionary to store the trained models in memory
trained_reordered_models = {}

print("=== Starting Fine-Tuning Pipeline for Task 1 (Reordering Strategies) ===")

for strat in strategies_to_train:
    print(f"\nLoading fresh base model for Task 1 | Strategy: {strat}...")

    # Load a clean instance of the base model for each strategy
    base_model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    trained_reordered_models[strat] = finetune_and_push_chessboard_model(
        strategy_name=strat,
        dataset=dataset_task1,
        processor=processor,
        model=base_model,
        peft_config=peft_config,
        task="task1",
        hf_org_prefix=hf_org_prefix,
    )

print("\nAll reordered models have been successfully trained and pushed to Hugging Face!")

Evaluating models using function **evaluate_chessboard_model_task_1()** defined in *src/eval/utilities.py*.

Models are downloaded from the hugging face repo.

In [ ]:
strategies_to_evaluate = ["zigzag", "spiral", "file_wise"]
hf_org_prefix = "bdatm-project"  # Assicurati che corrisponda al prefisso usato per il push

for strat in strategies_to_evaluate:
    print(f"\nLoading and evaluating SFT model for strategy: {strat.upper()} from Hugging Face...")

    # 1. Load fresh base model instance
    base_eval_model = AutoModelForImageTextToText.from_pretrained(
        "Qwen/Qwen3.5-0.8B",
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    # 2. Load the specific LoRA weights from Hugging Face Hub
    repo_id_source = f"{hf_org_prefix}/qwen-task1-{strat}-lora"
    model_to_eval = PeftModel.from_pretrained(base_eval_model, repo_id_source)

    # 3. Apply the specific patch reordering to the test split images
    reordered_test_split = dataset_task1["test"].map(
        lambda sample: {
            "image": reorder_chessboard_image(sample["image"], strategy=strat, grid_size=8)
        }
    )

    # 4. Run the evaluation utility function
    strat_results_df, strat_summary_df = evaluate_chessboard_model_task_1(
        model=model_to_eval,
        processor=processor,
        dataset_split=reordered_test_split,
        model_name=f"Qwen + LoRA ({strat.capitalize()} - SFT)"
    )

    # 5. Append results to the global comparison DataFrame
    all_models_results = pd.concat([all_models_results, strat_summary_df], ignore_index=True)

print("\n--- Final Comparative Summary Table (Loaded from Hub & Evaluated) ---")
display(all_models_results)

## Learned Reordering (Project Work)

Instead of a fixed strategy, trains a lightweight `PlackettLucePatchPolicy` (`src/training/learned_reordering.py`) jointly with LoRA to learn which 8x8 patch order helps the model most, via REINFORCE using the per-sample LM loss as the reward — see the module's docstring for how this differs from REOrder's own from-scratch-model approach.

**Known limitations of this first version, worth knowing before running:** unlike the `Trainer`-based cells above, this custom training loop has **no checkpointing or resume support** — a Colab disconnect mid-run loses all progress, not just the current epoch. It's also unbatched (one sample at a time, no gradient accumulation) and does two backward passes per sample (model + policy), so it's slower per-epoch than the standard fine-tuning above. `num_train_epochs` is kept low below (3, not the default 10) to keep a first run's wall-clock time and disconnect risk bounded — raise it once you've confirmed this runs end-to-end.

In [ ]:
print("=== Starting Learned Reordering Training Pipeline for Task 1 ===")

# Fresh base model instance, same pattern as the training-free strategies above
base_model_learned = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

learned_lora_model, learned_policy, repo_id_learned = train_with_learned_reordering(
    dataset=dataset_task1,
    processor=processor,
    model=base_model_learned,
    peft_config=peft_config,
    task="task1",
    hf_org_prefix=hf_org_prefix,
    num_train_epochs=3,  # kept low for a first run -- see the note above
)

print(f"\nLearned Reordering training complete. Model pushed to: {repo_id_learned}")

Evaluating the learned-reordering model: training samples permutations stochastically (that's how REINFORCE explores), but evaluation needs one fixed ordering — `policy.greedy_permutation()` gives the policy's single best guess (argsort of its learned logits, no sampling noise), which is what actually gets applied to the test set here.

In [ ]:
print("=== Evaluating the Learned Reordering model on Task 1 ===")

learned_permutation = learned_policy.greedy_permutation()
print(f"Learned (greedy) patch order: {learned_permutation}")

reordered_test_split = dataset_task1["test"].map(
    lambda sample: {
        "image": apply_patch_permutation(sample["image"], learned_permutation, grid_size=8)
    }
)

learned_results_df, learned_summary_df = evaluate_chessboard_model_task_1(
    model=learned_lora_model,
    processor=processor,
    dataset_split=reordered_test_split,
    model_name="Qwen + LoRA (Learned Reordering)"
)

all_models_results = pd.concat([all_models_results, learned_summary_df], ignore_index=True)

print("\n--- Comparative Summary Table (Including Learned Reordering) ---")
display(all_models_results)

Pushing Task 1 evaluation results to Hugging Face, so they can be reloaded later without re-running any inference (same approach used in *main.ipynb*)

In [ ]:
print("=== Pushing Task 1 Evaluation Results to Hugging Face ===")

hf_dataset_results_task1 = Dataset.from_pandas(all_models_results)
repo_id_eval_task1 = f"{hf_org_prefix}/evaluation-results-task1"

print(f"Pushing Task 1 evaluation results to Hugging Face Hub: {repo_id_eval_task1}...")
hf_dataset_results_task1.push_to_hub(
    repo_id_eval_task1,
    private=False
)
print("Task 1 evaluation results successfully pushed to Hugging Face Hub!")

Displaying oredered results

In [ ]:
print("=== Downloading Task 1 Evaluation Results from Hugging Face ===")
downloaded_eval_results_task1 = load_dataset(repo_id_eval_task1, split="train").to_pandas()

print("\n--- TASK 1 Comparative Summary Table (Ordered from Best to Worst by CER, loaded from Hugging Face) ---")
sorted_results_task1_from_hub = downloaded_eval_results_task1.sort_values(
    by="character_error_rate", ascending=True
).reset_index(drop=True)
display(sorted_results_task1_from_hub)

## Results and Final Considerations

Reloading the Task 1 evaluation results directly from Hugging Face (instead of from the in-memory `all_models_results`), to confirm they persist independently of this notebook session — same idea as the "Results and Final Considerations" section in *main.ipynb*.

In [ ]:
sorted_results_df = all_models_results.sort_values(by="character_error_rate", ascending=True).reset_index(drop=True)

print("\n--- Comparative Summary Table (Ordered from Best to Worst by CER) ---")
display(sorted_results_df)